# ಪಾಠ 10 - ಉತ್ಪಾದನೆಯಲ್ಲಿ AI ಏಜೆಂಟ್ಸ್  

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ (Python) ಬಳಸಿ AI ಏಜೆಂಟ್‌ಗಳ **ಉತ್ಪಾದನಾ ಮಾದರಿಗಳನ್ನು** ಕಲಿಯೋಣ. ನಾವು ಕವರ್ ಮಾಡುತ್ತಿದ್ದೇವೆ:  

- **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಕಾಲಮಾನ ಮತ್ತು ಲಾಗಿಂಗ್ ಸೇರಿಸುವುದು  
- **ಮೌಲ್ಯಮಾಪನ** — ಪ್ರತಿಕ್ರಿಯೆ ಗುಣಮಟ್ಟಕ್ಕೆ ಅಂಕ ಹಾಕಲು ಮೌಲ್ಯಮಾಪಕ ಏಜಂಟ್ ಬಳಸಿ  
- **ಖರ್ಚು ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಆಪ್ಟಿಮೈಸೇಶನ್ ಮತ್ತು ಮಾದರಿ ಆಯ್ಕೆಗಾಗಿ ರಣನೀತಿ  

ದೃಷ್ಠಾಂತವು ಬಳಕೆದಾರರಿಗೆ ಪ್ರಯಾಣ ಯೋಜಿಸಲು ಸಹಾಯ ಮಾಡುವ **ಪ್ರಯಾಣ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಮೇಲ್ನೋಟ ಮತ್ತು ಮೌಲ್ಯಮಾಪನ ಅದರಲ್ಲಿ ಅನ್ವಯಿಸಲಾಗಿದೆ.  


## ಸೆಟ್ ಅಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಉತ್ಪಾದನಾ ಪರಿಗಣನೆಗಳು

ಎಐ ಏಜೆಂಟ್‌ಗಳನ್ನು ಪ್ರೋಟೋಟೈಪ್‌ಗಳಿಂದ ಉತ್ಪಾದನೆಗೆ ಕರೆತರಲು ಮೂರು ಪ್ರಮುಖ ಅಂಶಗಳನ್ನು ಸಮಗ್ರವಾಗಿ ಗಮನಿಸಬೇಕು:

1. **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಏನು ಮಾಡುತ್ತಿದೆ, ಅದಕ್ಕೆ ಎಷ್ಟು ಸಮಯ ಬೇಕಾಗುತ್ತದೆ ಮತ್ತು ಯಾವ ಉಪಕರಣಗಳನ್ನು ಕಾಲ್ ಮಾಡುತ್ತಿದೆ ಎಂಬುದರ ಸ್ಪಷ್ಟ ದೃಷ್ಟಿ ಬೇಕು. ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಲಾಗಿಂಗ್ ಇಲ್ಲದೆ, ಉತ್ಪಾದನಾ ಸಮಸ್ಯೆಗಳನ್ನು ಡಿಬಗ್ಗಿಂಗ್ ಮಾಡುವುದು ಬಹಳ ಕಷ್ಟವಾಗುತ್ತದೆ.

2. **ಮೌಲ್ಯಮಾಪನ** — ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ಪರಿಶೀಲನೆಗಳು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಸಮಯದೊಂದಿಗೆ ನಿಖರ, ಸಂಪೂರ್ಣ ಮತ್ತು ಸಹಾಯಕವಾಗಿರುವುದನ್ನು ಖಚಿತಪಡಿಸುತ್ತವೆ. ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ನಿರ್ದಿಷ್ಟ ಮಾಡಿದ ಮಾನದಂಡಗಳ ವಿರುದ್ಧ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಅಂಕಿಸುತ್ತಬಹುದು.

3. **ಖರ್ಚು ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಬಳಕೆ ನೇರವಾಗಿ ಖರ್ಚುಗಳ ಮೇಲೆ ಪ್ರಭಾವ ಬೀರುತ್ತದೆ. ಪ್ರಾಂಪ್ಟ್ ಆಪ್ಟಿಮೈಝೇಶನ್, ಮಾದರಿ ಆಯ್ಕೆ ಮತ್ತು ಕ್ಯಾಶಿಂಗ್ ಎಂಬ ತಂತ್ರಗಳು ಗುಣಮಟ್ಟವನ್ನು ಹಾಳು ಮಾಡದೇ ಖರ್ಚುಗಳನ್ನು ನಿಯಂತ್ರಣದಲ್ಲಿಡಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


## ಒಂದು ಪ್ರೇಕ್ಷಣೀಯ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುವುದು

ನಾವು ಪ್ರವಾಸ ಉಪಕರಣಗಳನ್ನು నిర్వಚಿಸಿ ಏಜೆಂಟ್ ಕರೆ ಸಮಯದೊಂದಿಗೆ ಮುಟ್ಟಿ ನೋಡುವLatency ಅನ್ನು ಗಮನಿಸಬಹುದು. ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು OpenTelemetry ಅಥವಾ ಸಮಾನವಾದ ಟ್ರೇಸಿಂಗ್ ಬ್ಯಾಕ್‌ಎಂಡ್‌ನೊಂದಿಗೆ ಏಕೀಕರಿಸುತ್ತೀರಿ.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ಮೌಲ್ಯಮಾಪನ ಮಾದರಿಗಳು

ಸಾಮಾನ್ಯ ಉತ್ಪಾದನಾ ಮಾದರಿ ಎಂದರೆ ಎರಡನೇ ಏಜೆಂಟ್ ಅನ್ನು **ಮೌಲ್ಯಮಾಪಕ** ಎಂದು ಉಪಯೋಗಿಸುವುದು. ಮೌಲ್ಯಮಾಪಕ ಪ್ರಾಥಮಿಕ ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆ ಮುಂತಾದ ಪೂರ್ವನಿರ್ಧರಿತ ಮಾನದಂಡಗಳ ವಿರುದ್ಧ ಅಂಕೆಯನ್ನು ನೀಡುತ್ತದೆ.

ಇದರಿಂದ ಸಾಧ್ಯವಾಗುತ್ತದೆ:
- ಪ್ರತಿಕ್ರಿಯೆಗಳು ಬಳಕೆದಾರರಿಗೆ ತಲುಪುವ ಮೊದಲು ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ಗೇಟ್‌ಗಳು
- ಪ್ರಾಂಪ್ಟ್‌ಗಳು ಅಥವಾ ಮಾದರಿಗಳು ಬದಲಾಗುವಾಗ ಹಿಂಪಡೆಯುವಿಕೆ ಪತ್ತೆಹಚ್ಚುವಿಕೆ
- కాలಕಾಲಾಂತರ ಏಜೆಂಟ್ ಕಾರ್ಯಕ್ಷಮತೆಯ ನಿಗಾದ ವಿಷಯದ ನಿಗಾವಣೆ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ಖರ್ಚು ನಿರ್ವಹಣಾ ತಂತ್ರಗಳು

ಉತ್ಪಾದನಾ AI ಏಜೆಂಟ್‌ಗಳಿಗೆ ಖರ್ಚುಗಳನ್ನು ನಿಯಂತ್ರಿಸುವುದು ಅತ್ಯಂತ ಮಹತ್ವದ ವಿಷಯ. ಇಲ್ಲಿ ಪ್ರಮುಖ ತಂತ್ರಗಳು ಇವೆ:

| ತಂತ್ರ | ವಿವರಣೆ |
|---|---|
| **ಪ್ರಾಂಪ್ಟ್ ಆಪ್ಟಿಮೈಜೆಷನ್** | ವ್ಯವಸ್ಥೆಯ ಸೂಚನೆಗಳನ್ನು ಸಂಕ್ಷಿಪ್ತವಾಗಿರಿಸಿ. redundant context ತೆಗೆದು ಹಾಕಿ ಇನ್‌ಪುಟ್ ಟೋಕನ್ಗಳನ್ನು ಕಡಿಮೆಮಾಡಿ. |
    "| **ಮಾದರಿ ಆಯ್ಕೆ** | ಸರಳ ಕಾರ್ಯಗಳಿಗಾಗಿ (ಉದಾ: ವರ್ಗೀಕರಣ ಅಥವಾ ತೆಗೆಯುವಿಕೆ) ಸಣ್ಣ, ಕಡಿಮೆ ವೆಚ್ಚದ ಮಾದರಿಗಳನ್ನು (ಉದಾ: GPT-5-mini) ಬಳಸಿ, ಮತ್ತು ಜಟಿಲ ತರ್ಕಕ್ಕೆ ದೊಡ್ಡ ಮಾದರಿಗಳನ್ನು ಮೀಸಲಿಡಿ. |\n",
| **ಕ್ಯಾಶಿಂಗ್** | ಉಪಕರಣ ಫಲಿತಾಂಶಗಳನ್ನು ಮತ್ತು ಹೆಚ್ಚಾಗಿ ಕೇಳುವ ಪ್ರಶ್ನೆಗಳನ್ನು ಕ್ಯಾಶ್ ಮಾಡಿ ಪುನರಾವರ್ತಿತ API ಕರೆಗಳನ್ನು ತಪ್ಪಿಸಿ. |
| **ಟೋಕನ್ ಬಜೆಟ್‌ಗಳು** | ಅನಾಪೇಕ್ಷಿತವಾಗಿ ದೀರ್ಘ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ತಪ್ಪಿಸಲು `max_tokens` ಮಿತಿಗಳನ್ನು ನಿಗದಿಪಡಿಸಿ. |
| **ಬ್ಯಾಚಿಂಗ್** | ಸಾಧ್ಯವಾದರೆ ಹಲವು ಬಳಕೆದಾರ ಪ್ರಶ್ನೆಗಳನ್ನೊಂದು API ಕರೆಯೊಳಗಾಗಿ ಗುಂಪು ಮಾಡಿ. |

ಪ್ರಾಯೋಗಿಕವಾಗಿ, ಹಂತವಾರು ವಿಧಾನ ಉತ್ತಮವಾಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ: ಸರಳ ವಿನಂತಿಗಳನ್ನು ವೇಗವಾದ, ಕಡಿಮೆ ವೆಚ್ಚದ ಮಾದರಿಗೆ ಮಾರ್ಗದರ್ಶಿಸಿ ಮತ್ತು ಕೇವಲ ಜಟಿಲವಾದ ಪ್ರಶ್ನೆಗಳನ್ನು ಹೆಚ್ಚು ಶಕ್ತಿಶಾಲಿ ಮಾದರಿಗೆ ವರ್ಧಿಸಿ.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಕಲಿತಿರಿ:

1. ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯ ಮತ್ತು ಲಾಗಿಂಗ್ ಜೊತೆಗೆ **ನೋಟವಿಡುವಿಕೆಯನ್ನು ಸೇರಿಸುವುದು**, ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಮೇಲ್ವಿಚಾರಣೆಗೆ ನೆಲಸಿ ಹಾಕುವುದು.
2. ಪೂರ್ಣತೆ, ಸತ್ಯತೆ ಮತ್ತು ಸಹಾಯಕತೆ ಮೌಲ್ಯಮಾಪನ ಮಾಡುವ ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ಬಳಸಿ **ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಮೌಲ್ಯಮಾಪನ ಮಾಡುವುದು**.
3. ಪ್ರಾಂಪ್ಟ್ ಆಪ್ಟಿಮೈಸೇಶನ್, ಮಾದರಿ ಆಯ್ಕೆ, ಕ್ಯಾಶಿಂಗ್ ಮತ್ತು ಟೋಕನ್ ಬಜೆಟ್ ಗಳ ಮೂಲಕ **ಖರ್ಚುಗಳನ್ನು ನಿರ್ವಹಿಸುವುದು**.

ಈ ಉತ್ಪಾದನಾ ಮಾದರಿಗಳು ನಿಮ್ಮ AI ಏಜೆಂಟ್‌ಗಳು ವಿಶ್ವಾಸನೀಯ, ಅಳೆಯಬಹುದಾದ ಮತ್ತು ಖರ್ಚು-ಕಾರ್ಯಕ್ಷಮ ಆಗಿರುವುದನ್ನು ಖಚಿತಪಡಿಸುತ್ತವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
